# SKG Metrics

Builds the main heterogeneous knowledge graph (Paper / Author / Keyword) and a directed citation graph, then enriches every node with citation-derived, era-scoped, and author-level metrics.

**Input:** `output/metadata_enriched_after_openalex_year.csv`, `keywords_results/keywords_normalized_deduplicated_no_stopwords.csv`, `output/openalex_citations_cache.json`  
**Output:** `output/graph/graph_metrics_enriched.csv`, `output/graph/author_metrics.csv`, `output/graph/cross_era_flow.csv`, `output/graph/keyword_emergence.csv`, `output/graph/articles_leiden_clusters.html`

In [ ]:
from pathlib import Path
import ast
import colorsys
import itertools
import json
import time
from collections import defaultdict

import networkx as nx
import pandas as pd
from pyvis.network import Network
from tqdm import tqdm

OUTPUT_DIR = Path("..").resolve() / "output"
KW_DIR     = Path("..").resolve() / "keywords_results"
GRAPH_DIR  = OUTPUT_DIR / "graph"

META_PATH  = OUTPUT_DIR / "metadata_enriched_after_openalex_year.csv"
KW_PATH    = KW_DIR    / "keywords_normalized_deduplicated_no_stopwords.csv"
CACHE_PATH = OUTPUT_DIR / "openalex_citations_cache.json"

OUT_METRICS  = GRAPH_DIR / "graph_metrics_enriched.csv"
OUT_HTML     = GRAPH_DIR / "articles_leiden_clusters.html"
OUT_ERA_FLOW = GRAPH_DIR / "cross_era_flow.csv"
OUT_KW_EMG   = GRAPH_DIR / "keyword_emergence.csv"
OUT_AUTHORS  = GRAPH_DIR / "author_metrics.csv"

ERAS = ["1950-1977", "1977-1997", "1997-2017", "2017+"]


def normalize_fname(raw) -> str:
    """Normalize a pdf_file value to a lowercase filename without extension."""
    if pd.isna(raw):
        return ""
    s = str(raw).strip()
    return s[:-4].lower() if s.lower().endswith(".pdf") else s.lower()


def split_authors(raw) -> list:
    """Split a semicolon-separated author string into a list of names."""
    if pd.isna(raw):
        return []
    return [a.strip() for a in str(raw).split(";") if a.strip()]


def get_era(year) -> str:
    """Map a publication year to its research era bucket."""
    if pd.isna(year):
        return "unknown"
    y = int(year)
    if 1950 <= y < 1977:  
        return "1950-1977"
    if 1977 <= y < 1997:  
        return "1977-1997"
    if 1997 <= y < 2017:  
        return "1997-2017"
    if y >= 2017:          
        return "2017+"
    return "before_1950"


def pick_title(row) -> str:
    """Return the best available title, preferring Crossref over original."""
    for col in ("crossref_title", "original_title"):
        t = row.get(col)
        if isinstance(t, str) and t.strip():
            return t.strip()
    return str(row.get("pdf_file", "unknown"))


def pick_authors(row) -> list:
    """Return a list of author strings, preferring Crossref over original."""
    for col in ("crossref_authors", "original_authors"):
        a = row.get(col)
        if isinstance(a, str) and a.strip():
            return split_authors(a)
    return []


def parse_keywords(cell) -> list:
    """Return list of (keyword, strength_float) from a serialised keyword cell."""
    if pd.isna(cell):
        return []
    try:
        parsed = ast.literal_eval(str(cell).strip())
    except Exception:
        return []
    out = []
    for item in parsed:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            kw = str(item[0]).strip().lower()
            try:
                strength = float(item[1])
            except Exception:
                continue
            if kw:
                out.append((kw, strength))
    return out


def _norm_doi(doi) -> str:
    """Normalise a DOI string to a bare lowercase identifier."""
    if not isinstance(doi, str):
        return ""
    s = doi.replace("https://doi.org/", "").strip().lower()
    if "&" in s:
        s = s.split("&", 1)[0].strip()
    return s


def color_for_cluster(cluster_id: int) -> str:
    """Return a stable hex colour for a cluster id using golden-ratio hue steps."""
    if cluster_id < 0:
        return "#95a5a6"
    h = (cluster_id * 0.618033988749895) % 1.0
    s = 0.55 + 0.35 * ((cluster_id % 3) / 2.0)
    v = 0.75 + 0.20 * ((cluster_id % 2) / 1.0)
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))


# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
print("Loading data...")
meta = pd.read_csv(META_PATH)
kw_df = pd.read_csv(KW_PATH)

meta["filename"] = meta["pdf_file"].apply(normalize_fname)
meta["_year_combined"] = meta["crossref_year"].combine_first(meta["original_year"]).combine_first(meta.get("openalex_year", pd.Series(dtype=float)))
meta["era"] = meta["_year_combined"].apply(get_era)
meta["title"]    = meta.apply(pick_title, axis=1)
meta["authors"]  = meta.apply(pick_authors, axis=1)

KW_COL = "keywords_normalized" if "keywords_normalized" in kw_df.columns          else "keywords_without_stopwords"
kw_df["filename"]     = kw_df["filename"].apply(normalize_fname)
kw_df["keyword_items"] = kw_df[KW_COL].apply(parse_keywords)

fname_to_meta = {row["filename"]: row for _, row in meta.iterrows()}

doi_to_fname = {}
for _, row in meta.dropna(subset=["doi"]).iterrows():
    doi_to_fname[_norm_doi(str(row["doi"]))] = row["filename"]

# ---------------------------------------------------------------------------
# 2. Build main heterogeneous graph  (Paper / Author / Keyword)
# ---------------------------------------------------------------------------
print("\nBuilding main KG...")
G = nx.Graph()

for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Paper nodes"):
    G.add_node(
        row["filename"],
        type="Paper",
        title=row["title"],
        year=str(row.get("crossref_year", "")),
        era=row["era"],
    )

for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Author edges"):
    for author in row["authors"]:
        if not G.has_node(author):
            G.add_node(author, type="Author")
        G.add_edge(row["filename"], author, rel="AUTHORED")

kw_lookup = kw_df.set_index("filename")["keyword_items"].to_dict()
for fname, items in tqdm(kw_lookup.items(), desc="Keyword edges"):
    if fname not in G:
        continue
    agg = defaultdict(list)
    for kw, strength in items:
        agg[kw].append(strength)
    for kw, strengths in agg.items():
        avg = sum(strengths) / len(strengths)
        if not G.has_node(kw):
            G.add_node(kw, type="Keyword")
        G.add_edge(fname, kw, rel="HAS_KEYWORD", weight=avg)

print(f"Main KG — nodes: {G.number_of_nodes():,}  edges: {G.number_of_edges():,}")

# ---------------------------------------------------------------------------
# 3. Build directed citation graph from OpenAlex cache
# ---------------------------------------------------------------------------
print("\nBuilding citation graph from OpenAlex cache...")
G_cite = nx.DiGraph()

for _, row in meta.iterrows():
    G_cite.add_node(row["filename"])

with open(CACHE_PATH) as f:
    oa_cache = json.load(f)

oa_id_to_fname = {}
for doi, data in oa_cache.items():
    if doi in doi_to_fname and data.get("openalex_id"):
        oa_id_to_fname[data["openalex_id"]] = doi_to_fname[doi]

corpus_oa_ids = set(oa_id_to_fname)

added = 0
for doi, data in oa_cache.items():
    if doi not in doi_to_fname:
        continue
    src = doi_to_fname[doi]
    for ref_oa_id in data.get("referenced_works", []):
        if ref_oa_id in corpus_oa_ids:
            tgt = oa_id_to_fname[ref_oa_id]
            if tgt != src:
                G_cite.add_edge(src, tgt)
                added += 1

print(f"Citation graph — nodes: {G_cite.number_of_nodes():,}  edges: {added:,}")

# ---------------------------------------------------------------------------
# 4. Global metrics on main KG
# ---------------------------------------------------------------------------
print("\nComputing global metrics on main KG...")
degree_main = dict(G.degree())

t0 = time.time()
print("  PageRank (main KG)...")
pagerank_main = nx.pagerank(G, alpha=0.85, weight="weight")
print(f"    done in {time.time()-t0:.1f}s")

t0 = time.time()
print("  Betweenness centrality (k=200 samples)...")
betweenness_main = nx.betweenness_centrality(G, k=200, normalized=True, weight="weight", seed=42)
print(f"    done in {time.time()-t0:.1f}s")

# ---------------------------------------------------------------------------
# 5. Citation-derived metrics
# ---------------------------------------------------------------------------
print("\nComputing citation-derived metrics...")

in_degree_cite  = dict(G_cite.in_degree())
out_degree_cite = dict(G_cite.out_degree())

t0 = time.time()
print("  HITS (hubs & authorities)...")
hub_scores, authority_scores = nx.hits(G_cite, max_iter=500)
print(f"    done in {time.time()-t0:.1f}s")

t0 = time.time()
print("  Citation PageRank...")
pr_cite = nx.pagerank(G_cite, alpha=0.85)
print(f"    done in {time.time()-t0:.1f}s")

for node in G_cite.nodes():
    if G.has_node(node):
        G.nodes[node]["citation_count"] = in_degree_cite.get(node, 0)
        G.nodes[node]["references_made"] = out_degree_cite.get(node, 0)
        G.nodes[node]["authority"]       = round(authority_scores.get(node, 0), 8)
        G.nodes[node]["hub"]             = round(hub_scores.get(node, 0), 8)
        G.nodes[node]["citation_pr"]     = round(pr_cite.get(node, 0), 8)

# ---------------------------------------------------------------------------
# 6. Pioneer score
# ---------------------------------------------------------------------------
print("\nComputing pioneer scores...")

for paper in tqdm(G_cite.nodes(), desc="  Pioneer score", unit="paper"):
    if not G.has_node(paper):
        continue
    paper_era = G.nodes[paper].get("era", "")
    if paper_era not in ERAS:
        G.nodes[paper]["pioneer_score"]        = 0.0
        G.nodes[paper]["citations_same_era"]   = 0
        G.nodes[paper]["citations_later_eras"] = 0
        continue

    paper_era_idx = ERAS.index(paper_era)
    same_era, later_era = 0, 0

    for citing_paper, _ in G_cite.in_edges(paper):
        if not G.has_node(citing_paper):
            continue
        citing_era = G.nodes[citing_paper].get("era", "")
        if citing_era not in ERAS:
            continue
        citing_era_idx = ERAS.index(citing_era)
        if citing_era_idx == paper_era_idx:
            same_era += 1
        elif citing_era_idx > paper_era_idx:
            later_era += 1

    pioneer = later_era / (1 + same_era)
    G.nodes[paper]["pioneer_score"]        = round(pioneer, 4)
    G.nodes[paper]["citations_same_era"]   = same_era
    G.nodes[paper]["citations_later_eras"] = later_era

# ---------------------------------------------------------------------------
# 7. Cross-era citation flow matrix
# ---------------------------------------------------------------------------
print("\nBuilding cross-era flow matrix...")

flow = defaultdict(int)
for src, tgt in G_cite.edges():
    era_src = G.nodes[src].get("era") if G.has_node(src) else None
    era_tgt = G.nodes[tgt].get("era") if G.has_node(tgt) else None
    if era_src in ERAS and era_tgt in ERAS:
        flow[(era_src, era_tgt)] += 1

flow_rows = []
for era_src in ERAS:
    for era_tgt in ERAS:
        flow_rows.append({
            "citing_era": era_src,
            "cited_era":  era_tgt,
            "edge_count": flow[(era_src, era_tgt)],
        })
flow_df = pd.DataFrame(flow_rows)
flow_df.to_csv(OUT_ERA_FLOW, index=False)
print(f"  Saved -> {OUT_ERA_FLOW}")

# ---------------------------------------------------------------------------
# 8. Era-scoped PageRank
# ---------------------------------------------------------------------------
print("\nComputing era-scoped PageRank...")

paper_nodes = {n for n, d in G.nodes(data=True) if d.get("type") == "Paper"}

for era in tqdm(ERAS, desc="  Era-scoped PageRank", unit="era"):
    era_papers = {n for n in paper_nodes if G.nodes[n].get("era") == era}
    if len(era_papers) < 2:
        continue

    era_kw_nodes = set()
    for p in era_papers:
        for nb in G.neighbors(p):
            if G.nodes[nb].get("type") == "Keyword":
                era_kw_nodes.add(nb)

    sub = G.subgraph(era_papers | era_kw_nodes)
    era_pr = nx.pagerank(sub, alpha=0.85, weight="weight")

    attr_name = f"era_pr_{era.replace('-','_').replace('+','plus')}"
    for node, score in era_pr.items():
        if G.has_node(node):
            G.nodes[node][attr_name] = round(score, 8)

# ---------------------------------------------------------------------------
# 9. Keyword emergence scores
# ---------------------------------------------------------------------------
print("\nComputing keyword emergence scores...")

kw_era_strength = defaultdict(lambda: defaultdict(float))
kw_era_count    = defaultdict(lambda: defaultdict(int))

for _, row in kw_df.iterrows():
    fname = row["filename"]
    meta_row = fname_to_meta.get(fname)
    if meta_row is None:
        continue
    era = meta_row["era"]
    if era not in ERAS:
        continue
    for kw, strength in row["keyword_items"]:
        kw_era_strength[kw][era] += strength
        kw_era_count[kw][era]    += 1

emergence_rows = []
for kw in kw_era_strength:
    strengths = {era: kw_era_strength[kw].get(era, 0.0) for era in ERAS}
    counts    = {era: kw_era_count[kw].get(era, 0)      for era in ERAS}

    first_era = next((e for e in ERAS if strengths[e] > 0), None)
    peak_era  = max(ERAS, key=lambda e: strengths[e])

    first_strength = strengths[first_era] if first_era else 0
    last_strength  = strengths[ERAS[-1]]
    growth_rate    = (last_strength / first_strength) if first_strength > 0 else 0

    growth_by_transition = {}
    for i in range(1, len(ERAS)):
        prev, curr = ERAS[i - 1], ERAS[i]
        prev_s = strengths[prev]
        curr_s = strengths[curr]
        growth_by_transition[f"growth_{prev}_to_{curr}"] =             round((curr_s - prev_s) / (prev_s + 1e-9), 4)

    row_out = {
        "keyword":       kw,
        "first_era":     first_era,
        "peak_era":      peak_era,
        "growth_rate":   round(growth_rate, 4),
        **{f"strength_{e}": round(strengths[e], 4) for e in ERAS},
        **{f"count_{e}":    counts[e]              for e in ERAS},
        **growth_by_transition,
    }
    emergence_rows.append(row_out)

    if G.has_node(kw):
        G.nodes[kw]["first_era"]   = first_era or ""
        G.nodes[kw]["peak_era"]    = peak_era
        G.nodes[kw]["growth_rate"] = round(growth_rate, 4)

emergence_df = pd.DataFrame(emergence_rows)
emergence_df.sort_values("growth_rate", ascending=False, inplace=True)
emergence_df.to_csv(OUT_KW_EMG, index=False)
print(f"  Saved -> {OUT_KW_EMG}")

# ---------------------------------------------------------------------------
# 10. Author-level metrics
# ---------------------------------------------------------------------------
print("\nComputing author-level metrics...")

G_coauth = nx.Graph()
for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Co-authorship graph"):
    authors = row["authors"]
    for a1, a2 in itertools.combinations(authors, 2):
        if G_coauth.has_edge(a1, a2):
            G_coauth[a1][a2]["weight"] += 1
        else:
            G_coauth.add_edge(a1, a2, weight=1)

t0 = time.time()
print("  Co-author betweenness centrality...")
coauth_between = nx.betweenness_centrality(G_coauth, weight="weight", normalized=True)
print(f"    done in {time.time()-t0:.1f}s")

author_rows = []
author_nodes = [n for n, d in G.nodes(data=True)
                if d.get("type") == "Author" and "nan" not in n.lower()]

for author in tqdm(author_nodes, desc="Author metrics"):
    paper_neighbours = [nb for nb in G.neighbors(author)
                        if G.nodes[nb].get("type") == "Paper"]

    eras_active = sorted({G.nodes[p].get("era") for p in paper_neighbours
                          if G.nodes[p].get("era") in ERAS})
    era_indices = [ERAS.index(e) for e in eras_active]
    career_span = (max(era_indices) - min(era_indices)) if len(era_indices) > 1 else 0

    cite_counts = sorted(
        [in_degree_cite.get(p, 0) for p in paper_neighbours],
        reverse=True
    )
    h_index = sum(1 for i, c in enumerate(cite_counts, 1) if c >= i)
    total_citations = sum(in_degree_cite.get(p, 0) for p in paper_neighbours)

    pr_main   = pagerank_main.get(author, 0)
    co_btwn   = coauth_between.get(author, 0)

    author_rows.append({
        "author":             author,
        "paper_count":        len(paper_neighbours),
        "eras_active":        "|".join(eras_active),
        "career_span_eras":   career_span,
        "h_index":            h_index,
        "total_citations":    total_citations,
        "pagerank_main_kg":   round(pr_main, 8),
        "coauthor_betweenness": round(co_btwn, 8),
        "coauthor_degree":    G_coauth.degree(author) if G_coauth.has_node(author) else 0,
    })

    G.nodes[author]["h_index"]              = h_index
    G.nodes[author]["career_span_eras"]     = career_span
    G.nodes[author]["total_citations"]      = total_citations
    G.nodes[author]["coauthor_betweenness"] = round(co_btwn, 8)

author_df = pd.DataFrame(author_rows)
author_df.sort_values("h_index", ascending=False, inplace=True)
author_df.to_csv(OUT_AUTHORS, index=False)
print(f"  Saved -> {OUT_AUTHORS}")

# ---------------------------------------------------------------------------
# 11. Write final node-level metrics CSV
# ---------------------------------------------------------------------------
print("\nSaving enriched metrics CSV...")

metric_rows = []
for node, data in tqdm(G.nodes(data=True), total=G.number_of_nodes()):
    row = {
        "node":              node,
        "type":              data.get("type", ""),
        "era":               data.get("era", ""),
        "year":              data.get("year", ""),
        "degree":            degree_main.get(node, 0),
        "pagerank":          round(pagerank_main.get(node, 0), 8),
        "betweenness":       round(betweenness_main.get(node, 0), 8),
        "citation_count":    data.get("citation_count", 0),
        "references_made":   data.get("references_made", 0),
        "authority":         data.get("authority", 0),
        "hub":               data.get("hub", 0),
        "citation_pr":       data.get("citation_pr", 0),
        "pioneer_score":     data.get("pioneer_score", 0),
        "citations_same_era":   data.get("citations_same_era", 0),
        "citations_later_eras": data.get("citations_later_eras", 0),
        **{k: data.get(k, 0) for k in data if k.startswith("era_pr_")},
        "first_era":         data.get("first_era", ""),
        "peak_era":          data.get("peak_era", ""),
        "growth_rate":       data.get("growth_rate", 0),
        "h_index":                data.get("h_index", 0),
        "career_span_eras":       data.get("career_span_eras", 0),
        "total_citations":        data.get("total_citations", 0),
        "coauthor_betweenness":   data.get("coauthor_betweenness", 0),
    }
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(OUT_METRICS, index=False)
print(f"  Saved -> {OUT_METRICS}")

# ---------------------------------------------------------------------------
# 12. Pyvis visualization — Louvain-clustered paper graph
# ---------------------------------------------------------------------------
print("\nBuilding Pyvis visualization...")

G_cite_und = G_cite.to_undirected()
connected_papers = [n for n, deg in G_cite_und.degree() if deg > 0]
isolated_papers  = [n for n, deg in G_cite_und.degree() if deg == 0]
G_conn = G_cite_und.subgraph(connected_papers).copy()

communities = nx.community.louvain_communities(G_conn, weight=None, seed=42)
node_to_cluster = {}
for cid, nodes in enumerate(communities):
    for n in nodes:
        node_to_cluster[n] = cid
for n in isolated_papers:
    node_to_cluster[n] = -1

cluster_sizes = sorted((len(c) for c in communities), reverse=True)
print(f"  Louvain communities: {len(communities)}, largest: {cluster_sizes[0] if cluster_sizes else 0}")
print(f"  Connected papers: {len(connected_papers)}, isolated: {len(isolated_papers)}")

cluster_kw_agg = defaultdict(lambda: defaultdict(float))
for fname, items in kw_lookup.items():
    cid = node_to_cluster.get(fname, -1)
    if cid < 0:
        continue
    for kw, strength in items:
        cluster_kw_agg[cid][kw] += strength

cluster_label = {}
for cid, kw_agg in cluster_kw_agg.items():
    top3 = sorted(kw_agg.items(), key=lambda x: x[1], reverse=True)[:3]
    cluster_label[cid] = ", ".join(k for k, _ in top3) if top3 else f"cluster {cid}"
cluster_label[-1] = "isolated"

for n in G.nodes():
    if G.nodes[n].get("type") == "Paper":
        cid = node_to_cluster.get(n, -1)
        G.nodes[n]["cluster"] = cid
        G.nodes[n]["cluster_topic"] = cluster_label.get(cid, "unknown")

connected_set = set(connected_papers)
G_cite_vis = G_cite.subgraph(connected_set)

net = Network(height="950px", width="100%", bgcolor="white", font_color="black",
              notebook=False, directed=True)

net.barnes_hut(
    gravity=-42000,
    central_gravity=0.06,
    spring_length=260,
    spring_strength=0.012,
    damping=0.86,
    overlap=0.2,
)

for n in connected_papers:
    nd = G.nodes.get(n, {})
    cid = node_to_cluster.get(n, -1)
    cite_cnt = in_degree_cite.get(n, 0)
    color = color_for_cluster(cid)
    era = nd.get("era", "")
    title_text = nd.get("title", n)

    net.add_node(
        n,
        label=f"C{cid}",
        group=cid,
        title=(
            f"<b>{title_text}</b><br>"
            f"Cluster: C{cid} -- {cluster_label.get(cid, '')}<br>"
            f"Era: {era}<br>"
            f"Citations (in corpus): {cite_cnt}<br>"
            f"Pioneer score: {nd.get('pioneer_score', 0)}<br>"
            f"Authors: {nd.get('title', '')}"
        ),
        value=max(4, 4 + cite_cnt * 0.8),
        color=color,
    )

for u, v in G_cite_vis.edges():
    cu = node_to_cluster.get(u, -1)
    cv = node_to_cluster.get(v, -1)
    intra = cu == cv and cu >= 0
    net.add_edge(
        u, v,
        width=1.4 if intra else 0.4,
        color={"color": color_for_cluster(cu) if intra else "#666666",
               "opacity": 0.65 if intra else 0.12},
        arrows="to",
    )

net.set_options("""
var options = {
  "physics": {
    "enabled": true,
    "solver": "barnesHut",
    "stabilization": {"enabled": true, "iterations": 1800, "updateInterval": 25}
  },
  "nodes": {"font": {"size": 12}},
  "interaction": {"hover": true, "tooltipDelay": 150}
}
""")

net.write_html(str(OUT_HTML), open_browser=False)
print(f"  Saved -> {OUT_HTML}")
print(f"  Visualized nodes: {len(connected_papers)}, edges: {G_conn.number_of_edges()}")

# ---------------------------------------------------------------------------
# 13. Sanity check
# ---------------------------------------------------------------------------
print("\n=== SANITY CHECK ===")

papers = metrics_df[metrics_df["type"] == "Paper"]
authors_m = metrics_df[metrics_df["type"] == "Author"]
keywords_m = metrics_df[metrics_df["type"] == "Keyword"]

print(f"\nPapers:   {len(papers):,}")
print(f"Authors:  {len(authors_m):,}")
print(f"Keywords: {len(keywords_m):,}")

print("\nTop 10 papers by citation count:")
top_cited = papers[["node","era","citation_count","pioneer_score","authority"]] \
    .sort_values("citation_count", ascending=False).head(10)
print(top_cited.to_string(index=False))

print("\nTop 10 papers by pioneer score (cited by later eras):")
top_pioneer = papers[["node","era","pioneer_score","citation_count","citations_later_eras"]] \
    .sort_values("pioneer_score", ascending=False).head(10)
print(top_pioneer.to_string(index=False))

print("\nTop 10 authors by h-index:")
top_auth = author_df[["author","h_index","total_citations","career_span_eras","paper_count"]].head(10)
print(top_auth.to_string(index=False))

print("\nTop 10 fastest-growing keywords:")
top_kw = emergence_df[["keyword","first_era","peak_era","growth_rate"]].head(10)
print(top_kw.to_string(index=False))

print("\nCross-era citation flow matrix:")
pivot = flow_df.pivot(index="citing_era", columns="cited_era", values="edge_count").fillna(0).astype(int)
pivot = pivot.reindex(index=ERAS, columns=ERAS, fill_value=0)
print(pivot.to_string())

print("\nDone.")


Loading data...

Building main KG...


Keyword edges: 100%|██████████| 2366/2366 [00:00<00:00, 11128.61it/s]


Main KG — nodes: 39,534  edges: 71,156

Building citation graph from OpenAlex cache...
Citation graph — nodes: 2,376  edges: 3,014

Computing global metrics on main KG...
  PageRank (main KG)...
    done in 0.1s
  Betweenness centrality (k=200 samples)...
    done in 34.2s

Computing citation-derived metrics...
  HITS (hubs & authorities)...
    done in 0.0s
  Citation PageRank...
    done in 0.0s

Computing pioneer scores...


  Pioneer score: 100%|██████████| 2376/2376 [00:00<00:00, 407277.22paper/s]



Building cross-era flow matrix...
  Saved → /home/john/repos/leno4ka/output/graph/cross_era_flow.csv

Computing era-scoped PageRank...


...

    done in 1.9s


Author metrics: 100%|██████████| 3437/3437 [00:00<00:00, 253020.15it/s]


  Saved → /home/john/repos/leno4ka/output/graph/author_metrics.csv

Saving enriched metrics CSV...


100%|██████████| 39534/39534 [00:00<00:00, 402622.39it/s]


  Saved → /home/john/repos/leno4ka/output/graph/graph_metrics_enriched.csv

Building Pyvis visualization...
  Louvain communities: 26, largest: 208
  Connected papers: 883, isolated: 1493
  Saved → /home/john/repos/leno4ka/output/graph/articles_leiden_clusters.html
  Visualized nodes: 883, edges: 3006

=== SANITY CHECK ===

Papers:   2,376
Authors:  3,475
Keywords: 33,683

Top 10 papers by citation count:
                                              node       era  citation_count  pioneer_score  authority
50shannon programming a computer for playing chess 1950-1977              70        22.6667   0.007859
            96gobetsimon templates in chess memory 1977-1997              69        34.0000   0.047170
                                          18silver     2017+              56         0.0000   0.003619
                                  73simongilmartin 1950-1977              38        38.0000   0.018284
                 89schaeffer the history heuristic 1977-1997              36

## RQ2 — Influence Analysis

Computes a combined influence score per paper (authority + citation count + pioneer score + betweenness), ranks authors by cross-era impact, and identifies bridging keywords that connect research sub-fields.

In [ ]:
import pandas as pd
import networkx as nx
from collections import defaultdict

METRICS_PATH       = OUTPUT_DIR / "graph/graph_metrics_enriched.csv"
AUTHORS_PATH       = OUTPUT_DIR / "graph/author_metrics.csv"
RQ2_PAPERS_OUT     = OUTPUT_DIR / "graph/rq2_paper_influence.csv"
RQ2_ERA_OUT        = OUTPUT_DIR / "graph/rq2_era_top_papers.csv"
RQ2_AUTHORS_OUT    = OUTPUT_DIR / "graph/rq2_author_influence.csv"
RQ2_TECHNIQUES_OUT = OUTPUT_DIR / "graph/rq2_technique_influence.csv"

metrics   = pd.read_csv(METRICS_PATH)
author_df = pd.read_csv(AUTHORS_PATH)
papers    = metrics[metrics["type"] == "Paper"].copy()
keywords  = metrics[metrics["type"] == "Keyword"].copy()

# ------------------------------------------------------------------
# A. Exact betweenness on citation graph (paper-paper)
# ------------------------------------------------------------------
print("Computing exact betweenness on citation graph...")

if "G_cite" not in dir():
    meta_df = pd.read_csv(META_PATH)
    def _norm(x):
        if pd.isna(x): 
            return ""
        s = str(x).strip()
        return (s[:-4] if s.lower().endswith(".pdf") else s).lower()
    def _nd(doi):
        if not isinstance(doi, str):
            return ""
        s = doi.replace("https://doi.org/","").strip().lower()
        return s.split("&",1)[0].strip() if "&" in s else s
    meta_df["filename"] = meta_df["pdf_file"].apply(_norm)
    doi_to_fname = {_nd(str(r["doi"])): r["filename"] for _, r in meta_df.dropna(subset=["doi"]).iterrows()}
    with open(CACHE_PATH) as f:
        oa_cache = json.load(f)
    oa_id_to_fname = {d["openalex_id"]: doi_to_fname[doi]
                      for doi, d in oa_cache.items()
                      if doi in doi_to_fname and d.get("openalex_id")}
    corpus_oa = set(oa_id_to_fname)
    G_cite = nx.DiGraph()
    for _, r in meta_df.iterrows(): 
        G_cite.add_node(r["filename"])
    for doi, d in oa_cache.items():
        if doi not in doi_to_fname: 
            continue
        src = doi_to_fname[doi]
        for ref in d.get("referenced_works", []):
            if ref in corpus_oa:
                tgt = oa_id_to_fname[ref]
                if tgt != src:
                    G_cite.add_edge(src, tgt)

G_cite_und = G_cite.to_undirected()
betweenness_cite = nx.betweenness_centrality(G_cite_und, normalized=True)
print(f"  done -- {len(betweenness_cite)} nodes")

papers["betweenness_cite"] = papers["node"].map(betweenness_cite).fillna(0).round(6)

# ------------------------------------------------------------------
# B. Combined influence score
# ------------------------------------------------------------------
def minmax(s):
    """Normalise a pandas Series to [0, 1]."""
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn + 1e-12)

papers["influence_score"] = (
    minmax(papers["authority"])
  + minmax(papers["citation_count"])
  + minmax(papers["pioneer_score"])
  + minmax(papers["betweenness_cite"])
) / 4.0
papers["influence_score"] = papers["influence_score"].round(4)

papers.sort_values("influence_score", ascending=False).to_csv(RQ2_PAPERS_OUT, index=False)
print("\nTop 10 papers by combined influence score:")
cols = ["node","era","influence_score","citation_count","authority","pioneer_score","betweenness_cite"]
print(papers.sort_values("influence_score", ascending=False)[cols].head(10).to_string(index=False))

# ------------------------------------------------------------------
# C. Per-era ranked tables  (top 5 per era x 3 lenses)
# ------------------------------------------------------------------
era_rows = []
for era in ERAS:
    era_papers = papers[papers["era"] == era]
    for rank_col in ("authority", "citation_count", "pioneer_score", "influence_score"):
        top5 = era_papers.nlargest(5, rank_col)[["node", "era", rank_col]].copy()
        top5["rank_by"] = rank_col
        top5["rank"] = range(1, len(top5) + 1)
        era_rows.append(top5)

era_df = pd.concat(era_rows, ignore_index=True)
era_df.to_csv(RQ2_ERA_OUT, index=False)

print("\nPer-era top 3 by authority:")
for era in ERAS:
    top = papers[papers["era"] == era].nlargest(3, "authority")[["node","authority"]]
    print(f"  {era}: " + "  |  ".join(f"{r['node'][:35]} ({r['authority']:.4f})" for _, r in top.iterrows()))

# ------------------------------------------------------------------
# D. Technique influence -- keyword nodes ranked by KG betweenness
# ------------------------------------------------------------------
print("\nTop 20 bridging keywords (KG betweenness):")
kw_btwn = keywords[["node","betweenness","pagerank","growth_rate","peak_era","first_era"]].copy()
kw_btwn = kw_btwn.sort_values("betweenness", ascending=False)
kw_btwn.to_csv(RQ2_TECHNIQUES_OUT, index=False)
print(kw_btwn.head(20).to_string(index=False))

# ------------------------------------------------------------------
# E. Author cross-era influence  (career span x h-index x betweenness)
# ------------------------------------------------------------------
author_df["influence_score"] = (
    minmax(author_df["h_index"].astype(float))
  + minmax(author_df["total_citations"].astype(float))
  + minmax(author_df["coauthor_betweenness"].astype(float))
  + author_df["career_span_eras"].astype(float) / 3.0
) / 4.0
author_df["influence_score"] = author_df["influence_score"].round(4)
author_df.sort_values("influence_score", ascending=False).to_csv(RQ2_AUTHORS_OUT, index=False)

print("\nTop 10 authors by combined influence:")
print(author_df.sort_values("influence_score", ascending=False)
      [["author","influence_score","h_index","career_span_eras","total_citations"]].head(10).to_string(index=False))


Computing exact betweenness on citation graph...


/tmp/ipykernel_252790/599646313.py:21: DtypeWarning: Columns (0: era) have mixed types. Specify dtype option on import or set low_memory=False.
  metrics = pd.read_csv(METRICS_PATH)


  done — 2376 nodes

Top 10 papers by combined influence score:
                                              node       era  influence_score  citation_count  authority  pioneer_score  betweenness_cite
            96gobetsimon templates in chess memory 1977-1997           0.7780              69   0.047170        34.0000          0.006813
50shannon programming a computer for playing chess 1950-1977           0.6908              70   0.007859        22.6667          0.029411
                                  73simongilmartin 1950-1977           0.5389              38   0.018284        38.0000          0.006619
                                          18silver     2017+           0.3757              56   0.003619         0.0000          0.018411
                                        92charness 1977-1997           0.3664              35   0.020888         8.0000          0.009180
                               16burgoynesalagobet 1997-2017           0.3075              22   0.013279    